# Demo: HEP-multiagent + Science MCP Server Starter

This notebook installs HEP-multiagent and the science MCP server starter from GitHub, launches the MCP server over stdio, and sends one Argo-backed agent prompt.


## 0. Setup


In [ ]:
%pip install -q "git+ssh://git@github.com/HEP-KE/HEP-multiagent.git"


## 1. MCP Server Environment


In [ ]:
from pathlib import Path
import subprocess
import sys

runtime_dir = Path("demo_science_mcp_runtime").resolve()
mcp_env = runtime_dir / "mcp_env"
mcp_repo = runtime_dir / "science-mcp-server-starter"
runtime_dir.mkdir(parents=True, exist_ok=True)

if not mcp_repo.exists():
    subprocess.run(
        ["git", "clone", "git@github.com:HEP-KE/science-mcp-server-starter.git", str(mcp_repo)],
        check=True,
    )

subprocess.run([sys.executable, "-m", "venv", str(mcp_env)], check=True)
mcp_python = mcp_env / "bin" / "python"
subprocess.run(
    [
        str(mcp_python),
        "-m",
        "pip",
        "install",
        "-q",
        str(mcp_repo),
    ],
    cwd=mcp_repo,
    check=True,
)

print(mcp_python)


## 2. Argo LLM


In [ ]:
import os
from langchain_openai import ChatOpenAI

os.environ["ARGO_BASE_URL"] = "https://apps-dev.inside.anl.gov/argoapi/v1"
os.environ["ARGO_MODEL"] = "GPT-5.5"
os.environ["ARGO_USER"] = os.environ.get("ARGO_USER", "YOUR_ARGO_USERNAME")

if os.environ["ARGO_USER"] == "YOUR_ARGO_USERNAME":
    raise ValueError("Set ARGO_USER to your Argo username before running this cell.")

llm = ChatOpenAI(
    model=os.environ["ARGO_MODEL"],
    base_url=os.environ["ARGO_BASE_URL"],
    api_key=os.environ["ARGO_USER"],
)


## 3. Initialize Agent


In [ ]:
from hep_multiagent import Agent, AgentFeatures

mcp_servers = [
    {
        "name": "science-mcp",
        "transport": "stdio",
        "command": str(mcp_python),
        "args": ["-m", "mcp_server", "--transport", "stdio"],
        "cwd": str(mcp_repo),
    }
]

agent = Agent(
    llm=llm,
    mcp_servers=mcp_servers,
    features=AgentFeatures(
        report=True,
        citations=False,
        replay_notebook=True,
        execution_log=True,
    ),
)


## 4. Run


In [ ]:
result = await agent.run(
    "Use the MCP tools to generate 30 random points and create a sine wave plot with 30 sample points.",
    output_dir=str(runtime_dir / "agent_output"),
)

print(result.get("final_report", result))

from IPython.display import Image, display
display(Image(filename=runtime_dir / "agent_output" / "sine_wave.png"))
